# 12 CNN 反向传播与参数更新

前面已经学习了 CNN 的前向传播：图片经过卷积、激活、池化、展平、全连接层，最后得到类别分数和 loss。

现在要补上另一半：反向传播。

这一节不推复杂公式，只回答三个入门阶段最重要的问题：

```text
反向传播对谁求导？
修改哪些参数？
卷积核共享参数时，梯度怎么理解？
```

## 1. 先回顾：CNN 前向传播做了什么

CNN 前向传播可以先理解成：

```text
输入图片
-> 卷积层提取特征
-> 激活函数处理特征
-> 池化层压缩特征图
-> Flatten 展平
-> 全连接层输出类别分数
-> 计算 loss
```

前向传播的结果是得到一个 loss。

loss 越大，说明模型预测和真实标签差得越多。

训练的目标就是让 loss 变小。

## 2. 反向传播要解决什么问题

反向传播要解决的问题是：

```text
每个可学习参数应该怎么改，才能让 loss 变小？
```

这里的可学习参数，就是训练过程中会被更新的数字。

在 CNN 中，常见可学习参数包括：

- 卷积核里的权重。
- 卷积核对应的 bias。
- 全连接层里的权重。
- 全连接层里的 bias。

反向传播不是直接修改图片，也不是直接修改特征图。

它真正要计算的是：loss 对这些可学习参数的影响。

## 3. 对谁求导

反向传播的核心是求梯度。

梯度可以先理解成：

```text
某个参数变化一点点，loss 会怎么变。
```

假设 loss 叫 $L$。

卷积核里的某个权重叫 $w$。

那我们要求的是：

$$
\frac{\partial L}{\partial w}
$$

意思是：loss 对这个权重的导数。

如果卷积核有 bias，叫 $b$，也要求：

$$
\frac{\partial L}{\partial b}
$$

对全连接层的权重和 bias 也是一样。

## 4. 修改哪些参数

训练时会修改有可学习参数的层。

卷积层有参数，所以会更新：

```text
卷积核权重
卷积核 bias
```

全连接层有参数，所以会更新：

```text
全连接层权重
全连接层 bias
```

Flatten 通常没有参数，所以不更新参数。

池化层通常没有参数，所以不更新参数。

ReLU 这类激活函数通常也没有参数，所以不更新参数。

它们虽然不更新参数，但反向传播时梯度仍然会经过它们继续往前传。

## 5. 参数怎么更新

前面学习梯度下降时已经知道，参数更新的基本思想是：

$$
新参数 = 旧参数 - 学习率 \times 梯度
$$

如果是卷积核里的某个权重 $w$，可以写成：

$$
w_{new}=w_{old}-\eta\frac{\partial L}{\partial w}
$$

其中 $\eta$ 是学习率。

bias 也是一样：

$$
b_{new}=b_{old}-\eta\frac{\partial L}{\partial b}
$$

所以训练时真正被改动的，是权重和 bias 这些参数。

## 6. 卷积核不是一个神经元

这里要特别注意一个容易误会的地方。

一个卷积核不是一个神经元。

更准确地说：

```text
一个卷积核是一组共享参数。
```

卷积核在图片上滑动。

每滑到一个位置，就产生一个输出值。

这个位置上的输出值，可以类比成一个局部位置的神经元输出。

但是卷积核本身是一组会被很多位置重复使用的参数。

## 7. 卷积层的权重梯度怎么理解

卷积层反向传播最重要的一点是：同一个卷积核权重会在很多位置重复使用。

比如一个 3 x 3 卷积核里的某个权重 $w_1$。

前向传播时，它会在图片很多局部区域中被用到。

所以反向传播时，很多输出位置都会对 $w_1$ 产生影响。

最后，$w_1$ 的总梯度要把这些位置的影响加起来。

可以先这样理解：

```text
卷积核某个权重的梯度
=
它在所有滑动位置产生的梯度贡献之和
```

这就是权值共享在反向传播中的体现。

## 8. 为什么梯度要累加

因为同一个权重参与了多个位置的计算。

如果一个参数只在一个地方用过，那它只需要接收一个地方传来的影响。

但卷积核参数在整张图上重复使用。

所以它会收到很多位置传回来的影响。

比如：

```text
左上角位置说：这个权重让我这里的预测受到了影响。
中间位置说：这个权重也影响了我这里。
右下角位置说：这个权重也影响了我这里。
```

最后这些影响要合在一起，才能知道这个权重整体应该往哪个方向调整。

所以卷积核权重的梯度来自所有使用过它的位置。

## 9. bias 的梯度怎么理解

一个卷积核通常对应一个 bias。

这个 bias 会加到这个卷积核产生的特征图的每个位置上。

## 9. bias 的梯度怎么理解

一个卷积核通常对应一个 bias。

这个 bias 会加到这个卷积核产生的特征图的每个位置上。

也就是说，同一张特征图里的很多位置都用到了同一个 bias。

所以反向传播时，bias 的梯度也要把这些位置的影响加起来。

可以先记成：

```text
卷积层 bias 的梯度
=
对应特征图所有位置传回来的梯度之和
```

如果这一层有 16 个卷积核，通常就有 16 个 bias。

每个 bias 对应一张输出特征图。

## 10. 池化层反向传播做什么

池化层通常没有可学习参数。

所以池化层本身没有权重和 bias 要更新。

但反向传播时，梯度仍然要穿过池化层，继续传给前面的卷积层。

如果是最大池化，前向传播时每个小区域只保留最大值。

反向传播时，梯度通常只传回当时那个最大值所在的位置。

可以先这样理解：

```text
最大池化前向传播时谁最大，反向传播时梯度主要传给谁。
```

池化层不更新参数，但会决定梯度往前传到哪些位置。

```text
平均池化在反向传播时会把上游梯度“均匀分配”回池化窗口的每个输入位置（按通道独立处理）。

具体地，若窗口大小为 k，则 dL/dy = 上游梯度，且对窗口内每个元素 xi： dL/dxi = dL/dy * (1/k).

若窗口重叠（stride < kernel），某个输入会收到来自多个窗口的梯度之和。

然后这些作为对卷积输出（feature map）的梯度，继续用于计算对卷积核的梯度（与对应输入 patch 做相关运算）和对卷积层输入的反传。

与最大池化不同，平均池化是线性的、平滑分配梯度。
```

## 11. ReLU 反向传播做什么

ReLU 通常也没有可学习参数。

但它会影响梯度能不能继续往前传。

ReLU 前向传播的规则是：

```text
输入大于 0，就保留。
输入小于等于 0，就变成 0。
```

反向传播时可以先这样理解：

```text
前向时被保留的位置，梯度可以继续传。
前向时被压成 0 的位置，梯度通常传不过去。
```

所以 ReLU 像一个简单的门，控制哪些位置的梯度继续往前走。

## 12. Flatten 反向传播做什么

Flatten 没有可学习参数。

它前向传播只是把多维特征图拉成一维向量。

反向传播时，它做的事情也很简单：把梯度从一维向量形状还原回原来的特征图形状。

比如前向传播：

```text
32 x 7 x 7 -> 1568
```

反向传播时就是反过来整理形状：

```text
1568 -> 32 x 7 x 7
```

Flatten 不更新参数，只负责梯度形状能接回前面的卷积部分。

## 13. 全连接层反向传播

全连接层和前面学 MLP 时一样。

它有权重和 bias。

反向传播时，会计算：

$$
\frac{\partial L}{\partial W}
$$

以及：

$$
\frac{\partial L}{\partial b}
$$

然后优化器根据这些梯度更新全连接层参数。

所以 CNN 的后半部分，和 MLP 的反向传播是同一个思路。

## 14. CNN 反向传播的整体路线

前向传播是从输入走向 loss：

```text
输入图片
-> 卷积
-> ReLU
-> 池化
-> Flatten
-> 全连接
-> loss
```

反向传播就是从 loss 往回走：

```text
loss
-> 全连接层参数
-> Flatten 还原梯度形状
-> 池化层传回对应位置
-> ReLU 控制梯度能否通过
-> 卷积层权重和 bias
```

最后优化器根据梯度更新可学习参数。

## 15. 哪些东西会被更新，哪些不会

可以把 CNN 中常见部分分成两类。

会被更新的：

```text
卷积核权重
卷积层 bias
全连接层权重
全连接层 bias
```

通常不会被更新的：

```text
输入图片
特征图本身
ReLU
池化层
Flatten
loss 数值本身
```

特征图会参与计算，也会传递梯度。

但训练保存下来的不是特征图，而是模型参数。

## 16. 本节小结

这一节先记住这些规则：

1. CNN 反向传播的目标是计算 loss 对可学习参数的梯度。
2. 可学习参数主要包括卷积核权重、卷积层 bias、全连接层权重和全连接层 bias。
3. 一个卷积核不是一个神经元，而是一组共享参数。
4. 卷积核权重在多个位置重复使用，所以梯度来自所有使用位置的累加。
5. 卷积层 bias 也会接收对应特征图所有位置传回来的梯度。
6. 池化层、ReLU、Flatten 通常没有可学习参数，但梯度会经过它们继续往前传。
7. 优化器根据梯度更新权重和 bias。
8. CNN 训练和 MLP 训练的主线一样，都是让 loss 变小。

## 自检问题

1. CNN 反向传播主要对谁求导？
2. CNN 中哪些参数会被优化器更新？
3. 卷积核为什么不能简单理解成一个神经元？
4. 为什么卷积核权重的梯度要累加多个位置的贡献？
5. 卷积层 bias 的梯度来自哪里？
6. 池化层有没有可学习参数？
7. 最大池化反向传播时，梯度主要传给哪个位置？
8. ReLU 在反向传播中像什么？
9. Flatten 反向传播时主要做什么？
10. CNN 和 MLP 在训练主线上有什么相同点？